# 03r · 모듈 02–03 복습 — 닫힌 책 (Review of Modules 02–03, Closed Book)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e Ch. 2 (§2.1–2.4), Ch. 4 (§4.1–4.5), §3.2 (리틀/PASTA) — 복습 범위 전체, 막히는 문항만 해당 절을 연다 · Ross 12e §5.2–5.3 (지수·푸아송), §6.2–6.5, §6.8 (CTMC·균일화), §8.2–8.3, §8.9 (대기행렬) — 연습문제 보충 · Norris §2.1–2.8, §3.5–3.8, §5.2 — 증명 확인 |
| 선수 노트북 | 02a–02d (푸아송 과정), 03a–03f (CTMC·대기행렬·균일화) |
| 예상 소요 | 60분 |
| 상태 | draft |

이 노트북은 새 내용이 없다. 목적은 세 가지다.

1. 모듈 02–03의 핵심 정의·정리 12개를 **책 없이** 재구성하고 자기 채점한다.
2. 세 개의 핵심 결과 — 2상태 사슬의 $P(t)$, Erlang C/B, 검사 역설의 평균 — 를 **처음부터 유도**하고, 코드로 정확값과 시뮬레이션을 모두 확인한다.
3. 자기채점표로 약한 노트북을 찾아 캡스톤 C2 전에 보강한다.

규칙: 1절과 3절의 ✍️ 칸은 책과 이전 노트북을 **닫고** 채운다. 정답(`<details>`)은 채운 뒤에만 연다. 코드 셀은 유도가 끝난 뒤 실행해 "내 손 계산이 맞았는지"를 기계로 확인하는 용도다.

## 0. 준비 (Setup)

In [ ]:
import os
import heapq
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import transition_matrix, gillespie_paths, stationary_ctmc, mmc_metrics, birth_death_stationary
from spkit.poisson import poisson_arrivals, age_residual_at
from spkit.plots import plot_hist_vs_pdf

SEED, rng = rng_for("03r")
setup_plots()
N_PATHS = scale(8_000)     # R1: Gillespie 경로 수 (FAST: 800)
N_CUST = scale(100_000)    # R2: 손님 수 (FAST: 10_000)
N_REP = scale(20_000)      # R3: 푸아송 과정 반복 수 (FAST: 2_000)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_CUST={N_CUST}, N_REP={N_REP}")

## 1. 닫힌 책 — 12문항 (Closed-Book Recall)

각 문항의 답을 종이(또는 아래 ✍️ 셀)에 먼저 적으세요. 한 문항당 2분을 넘기지 말고, 막히면 X로 두고 넘어가세요 — 어디가 약한지 아는 것이 이 절의 목적입니다. 다 적은 뒤 정답을 펼쳐 5절의 자기채점표에 O / △ / X를 기록합니다.

**Q1.** (02a) 무기억성(memorylessness)을 식으로 쓰고, 그 성질을 갖는 유일한 연속분포는 무엇인가? 왜 유일한가(한 줄)?

<details><summary>정답</summary>

$P(X>s+t\mid X>s)=P(X>t)$; 지수분포. 생존함수가 $\bar F(s+t)=\bar F(s)\bar F(t)$를 만족하는데, 이는 코시 함수방정식(Cauchy functional equation)이라 단조(우연속) 해는 $e^{-\lambda t}$뿐이다.

</details>

**Q2.** (02a) 독립 $E_i\sim\mathrm{Exp}(\lambda_i)$, $i=1,\dots,m$일 때 $\min_i E_i$의 분포, $i$번 시계가 이길 확률, 그리고 둘의 관계는?

<details><summary>정답</summary>

$\min\sim\mathrm{Exp}(\sum_j\lambda_j)$, $P(i\text{ wins})=\lambda_i/\sum_j\lambda_j$, 그리고 $\min$의 값과 승자는 **독립**이다. (이것이 03a의 "체류시간과 점프 방향이 독립"의 뿌리다.)

</details>

**Q3.** (02b) 푸아송 과정의 세 동치 정의를 말하고, $N(T)=n$이 주어졌을 때 도착 시각들의 조건부 분포는?

<details><summary>정답</summary>

(i) 독립·정상 증분 + $N(t)\sim\mathrm{Poisson}(\lambda t)$; (ii) i.i.d. $\mathrm{Exp}(\lambda)$ 간격의 누적합; (iii) 미소 정의 $P(N(t+h)-N(t)=1)=\lambda h+o(h)$, $P(\ge2)=o(h)$ + 독립 증분. 조건부로 $(0,T]$ 위 $n$개 i.i.d. 균등의 **순서통계량**(order statistics).

</details>

**Q4.** (02c) 세분화(thinning)·중첩(superposition)의 결론과, 비율 $\lambda(t)$인 NHPP의 $N(t)$ 분포는?

<details><summary>정답</summary>

확률 $p$ 세분화 → 서로 **독립**인 푸아송 $(p\lambda)$, $((1-p)\lambda)$; 독립 푸아송의 합은 푸아송(비율 합); $N(t)\sim\mathrm{Poisson}(\Lambda(t))$, $\Lambda(t)=\int_0^t\lambda(s)\,ds$.

</details>

**Q5.** (02d) 복합 푸아송 $\sum_{i\le N(t)}Y_i$의 분산과, 고정 시각 $t$를 덮는 간격 길이의 평균은?

<details><summary>정답</summary>

$\mathrm{Var}=\lambda t\,E[Y^2]$ (평균은 $\lambda tE[Y]$); 덮는 간격 평균 $(2-e^{-\lambda t})/\lambda\to2/\lambda$ — 검사 역설(inspection paradox): 나이 $\sim\min(\mathrm{Exp}(\lambda),t)$, 잔여 $\sim\mathrm{Exp}(\lambda)$.

</details>

**Q6.** (03a) 생성원(generator) $Q$의 원소 뜻, 체류시간과 점프 사슬, 질레스피 한 스텝에 필요한 난수 개수는?

<details><summary>정답</summary>

$q_{ij}$ ($i\ne j$)는 $i\to j$ 비율, $Q_{ii}=-q_i=-\sum_{j\ne i}q_{ij}$, 행합 0; 체류 $\mathrm{Exp}(q_i)$, 점프 확률 $q_{ij}/q_i$, 둘은 독립; 난수 2개(체류 하나, 방향 하나).

</details>

**Q7.** (03b) 전진·후진 방정식과 그 해, 그리고 $P(t)$를 numpy로 계산할 때의 함정은?

<details><summary>정답</summary>

전진 $P'=PQ$, 후진 $P'=QP$, 해 $P(t)=e^{Qt}$ (행렬 지수). `np.exp(Q*t)`는 **원소별** 지수라 틀리다 — `scipy.linalg.expm(Q*t)`.

</details>

**Q8.** (03b/03c) 정상분포 조건 $\pi Q=0$의 의미와, 기약 유한 CTMC가 DTMC와 달리 수렴에 필요로 **하지 않는** 조건은?

<details><summary>정답</summary>

각 상태에서 들어오는 확률 흐름 = 나가는 흐름(전역 균형); $\pi P(t)=\pi\ \forall t$와 동치. 비주기성(aperiodicity)이 불필요 — $t>0$이면 $P(t)>0$이고, 균일화 사슬에는 자기 루프가 있다.

</details>

**Q9.** (03c) 출생-사망 정상분포의 곱 형태와 그 유도 한 줄, 그리고 M/M/∞의 정상분포는?

<details><summary>정답</summary>

$\pi_n=\pi_0\prod_{k<n}\lambda_k/\mu_{k+1}$; 절단 방정식 $\lambda_n\pi_n=\mu_{n+1}\pi_{n+1}$ (상태 $n|n+1$ 사이 벽을 넘는 양방향 흐름이 같다); M/M/∞는 $\mathrm{Poisson}(\lambda/\mu)$.

</details>

**Q10.** (03d) M/M/1의 $L,W$, 리틀의 법칙(Little's law)의 표본경로 증명 한 줄, PASTA의 뜻과 조건은?

<details><summary>정답</summary>

$L=\rho/(1-\rho)$, $W=1/(\mu-\lambda)$; $\int_0^TN(t)\,dt=\sum_{i:\ \text{도착}\le T}W_i$ ($T$에서 시스템이 비었을 때; 아니면 경계항이 $T$에 비해 무시됨 — 계단함수 아래 면적을 세로/가로로 세기); PASTA = 푸아송 도착이 보는 시스템 분포는 시간평균 분포와 같다 — 도착이 푸아송(미래와 무관)일 때만.

</details>

**Q11.** (03e) Erlang C와 Erlang B는 각각 무엇이며, 같은 총 용량의 빠른 서버 1대와 느린 서버 2대 중 $W$, $W_q$는 어느 쪽이 작은가?

<details><summary>정답</summary>

C = M/M/c의 대기 확률 $P(N\ge c)$; B = M/M/c/c의 차단 확률 (서비스 분포에 무감각, insensitivity). $W$는 빠른 1대가, $W_q$는 느린 2대가 작다.

</details>

**Q12.** (03f) 균일화(uniformization) 사슬의 정의, $P(t)$의 급수 표현, 절단 오차 한계는?

<details><summary>정답</summary>

$P=I+Q/\lambda$, $\lambda\ge\max_iq_i$; $P(t)=\sum_{n\ge0}e^{-\lambda t}\frac{(\lambda t)^n}{n!}P^n$; $N$항까지 절단 오차 $\le P(\mathrm{Poisson}(\lambda t)>N)$.

</details>

✍️ 내 답 (책을 덮고 Q1–Q12의 답을 여기에 적은 뒤 정답을 펼치세요):

## 2. 치트시트 (Cheat Sheet) — 채점이 끝난 뒤에 펼치기

1절을 채점한 뒤에만 여세요. 3절의 재유도 중 막히면 여기로 돌아옵니다.

<details><summary>치트시트 (모듈 02–03)</summary>

**지수/푸아송 (02a–02d).** $\min$ of $\mathrm{Exp}(\lambda_i)\sim\mathrm{Exp}(\sum\lambda_i)$, 승자 확률 $\lambda_i/\sum\lambda_j$, 둘은 독립. $N(t)\sim\mathrm{Poisson}(\lambda t)$. 세분화 → 독립 푸아송. 조건부 도착 = 균등 순서통계량. 복합 푸아송 $\mathrm{Var}=\lambda tE[Y^2]$. 덮는 간격 평균 $(2-e^{-\lambda t})/\lambda$.

**CTMC (03a–03b).** $Q$ 행합 0; 체류 $\mathrm{Exp}(q_i)$, 점프 $q_{ij}/q_i$. $P(t)=e^{Qt}$, $P'=PQ=QP$. $\pi Q=0$. 기약 유한 ⇒ $P(t)\to\mathbf 1\pi$ (주기 조건 없음). 점프 사슬 정상분포 $\nu$와의 관계 $\pi_i\propto\nu_i/q_i$.

**출생-사망/대기행렬 (03c–03e).** $\pi_n=\pi_0\prod_{k<n}\lambda_k/\mu_{k+1}$. M/M/1: $(1-\rho)\rho^n$, $L=\rho/(1-\rho)$, $W=1/(\mu-\lambda)$, 체류시간 $\sim\mathrm{Exp}(\mu-\lambda)$. M/M/∞: $\mathrm{Poisson}(\lambda/\mu)$. M/M/c ($a=\lambda/\mu$, $\rho=a/c$): Erlang C $=\pi_0\frac{a^c}{c!(1-\rho)}$, $L_q=C\frac{\rho}{1-\rho}$, $L=L_q+a$. Erlang B $=\frac{a^c/c!}{\sum_{n\le c}a^n/n!}$. $L=\lambda W$ (리틀). PASTA.

**균일화 (03f).** $P=I+Q/\lambda$, $P(t)=\sum_nw_nP^n$ ($w_n$ = 푸아송 가중치), 오차 $\le$ 푸아송 꼬리, 시뮬레이션 = $\mathrm{Poisson}(\lambda t)$개의 DTMC 스텝.

</details>

## 3. 재유도 — 세 가지 핵심 결과 (Re-derivations)

세 결과를 각각 **처음부터** 유도하세요. 각 결과마다 (i) 명제와 "가정이 왜 필요한가", (ii) ✍️ 내 유도 칸, (iii) 기대 답안(`<details>`), (iv) 정확값·시뮬레이션 코드 검사가 이어집니다. 코드는 유도가 끝난 뒤 실행합니다. 코드 검사가 실패하면 시뮬레이션이 아니라 **내 손 계산**을 먼저 의심하세요 — 정확값은 코드가 공식에서 계산하므로 유도의 상수 하나가 틀리면 바로 드러납니다.

### R1 재유도: 2상태 사슬의 $P(t)$

**명제.** 상태 $\{0,1\}$, $0\to1$ 비율 $\alpha$, $1\to0$ 비율 $\beta$인 CTMC에서

$$P_{01}(t)=\frac{\alpha}{\alpha+\beta}\bigl(1-e^{-(\alpha+\beta)t}\bigr).$$

이를 (a) 미소 논법(ODE)과 (b) $Q$의 고윳값 분해, 두 가지로 유도하라. 그리고 $\alpha=1,\beta=2,t=0.5$의 값을 손으로 계산하라.

**가정이 왜 필요한가.**
- 시간 동질·마르코프(반군 성질 $P(t+h)=P(t)P(h)$)가 있어야 $h\to0$ 미소 논법에서 ODE가 나온다.
- 유한 상태라 $Q$가 유계이고 $P'=PQ$, $P(0)=I$의 해가 $e^{Qt}$로 유일하다.

✍️ 내 유도 (R1): (책을 덮고 여기에 두 가지 유도와 $P_{01}(0.5)$의 값을 적으세요)

In [ ]:
# 여기에 풀이: 손으로 계산한 P_01(0.5)를 적어 두고 아래 코드 검사와 비교

<details><summary>기대 답안 (R1)</summary>

**(a) ODE.** $p(t)=P_{01}(t)$라 하자. $[t,t+h]$에서 상태 0은 확률 $\alpha h+o(h)$로 1로 가고, 상태 1은 확률 $\beta h+o(h)$로 0으로 간다(반군 성질로 $t$ 이후의 진행이 $t$의 상태에만 의존). 따라서
$p(t+h)=p(t)(1-\beta h)+(1-p(t))\alpha h+o(h)$, 즉
$$p'=\alpha-(\alpha+\beta)p,\qquad p(0)=0.$$
선형 ODE의 해는 $p(t)=\frac{\alpha}{\alpha+\beta}+Ce^{-(\alpha+\beta)t}$, $p(0)=0$에서 $C=-\frac{\alpha}{\alpha+\beta}$. (전진 방정식 $P'=PQ$의 $(0,1)$ 성분을 그대로 쓴 것과 같다 — 03b 정리 2.)

**(b) 고윳값 분해.** $Q=\begin{pmatrix}-\alpha&\alpha\\\beta&-\beta\end{pmatrix}$의 고윳값은 $0$과 $-(\alpha+\beta)$ (대각합 $=-(\alpha+\beta)$, 행렬식 $0$). 고윳값 0의 왼쪽 고유벡터가 정상분포 $\pi=\left(\frac{\beta}{\alpha+\beta},\frac{\alpha}{\alpha+\beta}\right)$, 오른쪽 고유벡터가 $\mathbf 1$. 스펙트럼 사영으로
$$e^{Qt}=\mathbf 1\pi+e^{-(\alpha+\beta)t}(I-\mathbf 1\pi),$$
$(0,1)$ 성분: $\pi_1(1-e^{-(\alpha+\beta)t})=\frac{\alpha}{\alpha+\beta}(1-e^{-(\alpha+\beta)t})$. (검산: $\mathbf 1\pi$와 $I-\mathbf 1\pi$가 서로 소멸하는 사영이고 $Q=-(\alpha+\beta)(I-\mathbf 1\pi)$임을 확인하면 된다.)

**수치.** $\alpha=1,\beta=2$: $Q=\begin{pmatrix}-1&1\\2&-2\end{pmatrix}$, 고윳값 $0,-3$, $\pi=(2/3,1/3)$, $P_{01}(t)=\frac13(1-e^{-3t})$, $P_{01}(0.5)=\frac13(1-e^{-1.5})\approx0.258957$.

</details>

코드 검사 1 — 정확값: `transition_matrix`(= `expm`)와 닫힌 형식, 그리고 고윳값 분해로 만든 행렬을 $10^{-10}$ 수준에서 비교한다.

In [ ]:
ALPHA, BETA, T1 = 1.0, 2.0, 0.5
Q2 = np.array([[-ALPHA, ALPHA], [BETA, -BETA]])
p01_exact = ALPHA / (ALPHA + BETA) * (1 - np.exp(-(ALPHA + BETA) * T1))   # 닫힌 형식 (유도 결과)

P_expm = transition_matrix(Q2, T1)                                          # scipy expm
pi2 = stationary_ctmc(Q2)                                                   # (2/3, 1/3)
P_eig = np.outer(np.ones(2), pi2) + np.exp(-(ALPHA + BETA) * T1) * (np.eye(2) - np.outer(np.ones(2), pi2))
print(f"P_01(0.5): expm = {P_expm[0, 1]:.10f}, closed form = {p01_exact:.10f}, eigen = {P_eig[0, 1]:.10f}")
np.testing.assert_allclose(P_expm[0, 1], p01_exact, rtol=1e-10)
np.testing.assert_allclose(P_eig, P_expm, atol=1e-12)
print("R1 exact check: ok")

코드 검사 2 — 시뮬레이션: 질레스피 경로 $N$개를 상태 0에서 출발시켜 $t=0.5$에 상태 1에 있는 비율을 센다(03a 단계 4 재현). 곡선 전체도 $\frac13(1-e^{-3t})$와 겹쳐 본다.

In [ ]:
t_grid, X1 = gillespie_paths(Q2, 0, 2.0, rng, N_PATHS, n_grid=201)
idx = int(np.argmin(np.abs(t_grid - T1)))
r1_hat = mc_proportion(X1[:, idx] == 1)
print(f"t_grid[idx] = {t_grid[idx]:.3f}, fraction in state 1: {r1_hat}  (exact {p01_exact:.6f})")

fig, ax = plt.subplots()
ax.plot(t_grid, X1.mean(axis=0), label=f"Gillespie fraction in state 1 ({N_PATHS} paths)")
ax.plot(t_grid, ALPHA / (ALPHA + BETA) * (1 - np.exp(-(ALPHA + BETA) * t_grid)), "--", label=r"$\frac{1}{3}(1-e^{-3t})$")
ax.axhline(ALPHA / (ALPHA + BETA), color="gray", lw=0.8, label=r"$\pi_1 = 1/3$")
ax.set(xlabel="t", ylabel=r"$P_{01}(t)$", title="Two-state chain: P(X_t = 1 | X_0 = 0)")
ax.legend();

### R2 재유도: Erlang C와 Erlang B

**명제.** M/M/c ($a=\lambda/\mu$, $\rho=a/c<1$)의 곱 형태에서 $\pi_0$, Erlang C $=C(c,a)=P(N\ge c)$, $L_q$를 유도하고, M/M/c/c에서 Erlang B $=B(c,a)=\pi_c$를 유도하라. $c=3,a=2$에서 $C=4/9$, $L_q=8/9$, $W_q=4/9$, $B=4/19$를 **손으로** 계산하라. 보너스: 재귀식 $B(k)=\dfrac{aB(k-1)}{k+aB(k-1)}$, $B(0)=1$을 증명하라.

**가정이 왜 필요한가.**
- $\rho=a/c<1$이어야 기하급수 꼬리가 수렴해 $\pi_0>0$, 즉 Erlang C가 존재한다.
- 손실 시스템(M/M/c/c)은 상태가 유한이라 안정성 조건이 필요 없다 — $a>c$여도 $B$는 잘 정의된다.
- PASTA가 시간평균 $P(N\ge c)$를 **도착하는 손님이 보는** 대기 확률로 바꿔 준다. 시뮬레이션에서 "손님 중 기다린 비율"을 세는 것이 정당한 이유다.

✍️ 내 유도 (R2): (책을 덮고 여기에 곱 형태부터 $C, L_q, W_q, B$까지, 그리고 $c=3,a=2$의 손 계산을 적으세요)

In [ ]:
# 여기에 풀이: 손으로 계산한 C(3,2), L_q, W_q, B(3,2)를 적어 두고 아래 코드 검사와 비교

<details><summary>기대 답안 (R2)</summary>

**곱 형태.** M/M/c는 $\lambda_n=\lambda$, $\mu_n=\min(n,c)\mu$인 출생-사망 사슬. 절단 방정식에서
$$\pi_n=\pi_0\frac{a^n}{n!}\ (n\le c),\qquad \pi_n=\pi_0\frac{a^c}{c!}\rho^{\,n-c}\ (n\ge c).$$

**$\pi_0$와 Erlang C.** $\sum_n\pi_n=1$에서
$\pi_0^{-1}=\sum_{n<c}\frac{a^n}{n!}+\frac{a^c}{c!}\cdot\frac1{1-\rho}$. 대기 확률은 (PASTA로 도착 관점 = 시간평균)
$$C(c,a)=P(N\ge c)=\sum_{n\ge c}\pi_n=\pi_0\frac{a^c}{c!(1-\rho)}.$$

**$L_q$.** $L_q=\sum_{n\ge c}(n-c)\pi_n=\pi_0\frac{a^c}{c!}\sum_{k\ge0}k\rho^k=\pi_0\frac{a^c}{c!}\cdot\frac{\rho}{(1-\rho)^2}=C\cdot\frac{\rho}{1-\rho}$. 리틀: $W_q=L_q/\lambda$, $L=L_q+a$ (서비스 중인 손님 평균 $=a$, $L_q+c$가 **아님**).

**Erlang B.** M/M/c/c는 같은 사슬을 $c$에서 절단: $\pi_n=\pi_0a^n/n!$, $n\le c$, 정규화하면
$$B(c,a)=\pi_c=\frac{a^c/c!}{\sum_{n\le c}a^n/n!}.$$
PASTA로 도착 손님이 차단될 확률이 $\pi_c$다. (이 값은 서비스 분포에 무감각 — 03e.)

**손 계산 $c=3,a=2$.** $\rho=2/3$. $\sum_{n<3}a^n/n!=1+2+2=5$, $\frac{a^3}{3!}=\frac86=\frac43$, $\frac{a^3}{3!(1-\rho)}=\frac43\cdot3=4$. $\pi_0=1/(5+4)=1/9$. $C=4\cdot\frac19=\frac49$. $L_q=\frac49\cdot\frac{2/3}{1/3}=\frac89$. $W_q=L_q/\lambda=\frac89/2=\frac49$ (단위: $1/\mu$). $B(3,2)=\frac{4/3}{5+4/3}=\frac{4/3}{19/3}=\frac4{19}$.

**보너스 (재귀).** $S_k=\sum_{n\le k}\frac{a^n}{n!}$이라 하면 $B(k)^{-1}=\frac{k!}{a^k}S_k=\frac{k!}{a^k}\left(S_{k-1}+\frac{a^k}{k!}\right)=1+\frac ka\cdot\frac{(k-1)!}{a^{k-1}}S_{k-1}=1+\frac ka B(k-1)^{-1}$. 역수를 취해 정리하면 $B(k)=\frac{aB(k-1)}{k+aB(k-1)}$. 검산: $B(0)=1$, $B(1,2)=\frac{2}{1+2}=\frac23$, $B(2,2)=\frac{2\cdot2/3}{2+4/3}=\frac{4/3}{10/3}=\frac25$, $B(3,2)=\frac{2\cdot2/5}{3+4/5}=\frac{4/5}{19/5}=\frac4{19}$ ✓. (팩토리얼 없이 $c$가 수백이어도 안정적으로 계산하는 방법이다.)

</details>

코드 검사 1 — 정확값: `mmc_metrics`(Erlang C 공식)와 `birth_death_stationary`(곱 형태를 직접 정규화)로 M/M/3/3의 $\pi_3$, 그리고 재귀식을 서로 비교한다. 유도한 분수와 $10^{-10}$ 수준에서 일치해야 한다.

In [ ]:
LAM, MU, C_SRV = 2.0, 1.0, 3
a = LAM / MU
m = mmc_metrics(LAM, MU, C_SRV)
pi_333 = birth_death_stationary(np.full(C_SRV, LAM), np.arange(C_SRV + 1) * MU)   # M/M/3/3: birth λ, death nμ
B_rec = 1.0                                                                       # 재귀 B(0) = 1
for k in range(1, C_SRV + 1):
    B_rec = a * B_rec / (k + a * B_rec)

erlang_c_hand, Lq_hand, Wq_hand, erlang_b_hand = 4 / 9, 8 / 9, 4 / 9, 4 / 19       # 손 계산 (기대 답안)
print(f"Erlang C: mmc_metrics = {m['p_wait']:.10f}   hand = {erlang_c_hand:.10f}")
print(f"L_q     : mmc_metrics = {m['Lq']:.10f}   hand = {Lq_hand:.10f}")
print(f"W_q     : mmc_metrics = {m['Wq']:.10f}   hand = {Wq_hand:.10f}")
print(f"Erlang B: product form = {pi_333[3]:.10f}, recursion = {B_rec:.10f}, hand = {erlang_b_hand:.10f}")
np.testing.assert_allclose([m["p_wait"], m["Lq"], m["Wq"]], [erlang_c_hand, Lq_hand, Wq_hand], rtol=1e-10)
np.testing.assert_allclose([pi_333[3], B_rec], [erlang_b_hand, erlang_b_hand], rtol=1e-10)
print("R2 exact check: ok")

코드 검사 2 — 시뮬레이션: 03e의 손님 단위 시뮬레이터(가장 먼저 비는 서버 힙)로 M/M/3의 "기다린 손님 비율"과 M/M/3/3의 "차단된 손님 비율"을 잰다. 손님 단위 표본은 **자기상관**이 있으므로 SE는 배치 평균으로 계산한다(03d) — 이것이 아래 trap 카드의 첫 항목이다.

In [ ]:
# helper (spkit에 없어서 여기서 정의) — 03e와 동일
def mmck_customers(lam, mu, c, n_customers, rng, K=None, service=None):
    """FIFO M/M/c (K=None) 또는 M/M/c/K 손실/유한용량 시스템의 손님 단위 시뮬레이션.
    가장 먼저 비는 서버에 배정 (힙). returns (A, start, D); 차단된 손님은 start=D=nan.
    service: None이면 Exp(1/mu); 아니면 service(n, rng) -> 서비스시간 배열."""
    A = np.cumsum(rng.exponential(1.0 / lam, n_customers))
    S = rng.exponential(1.0 / mu, n_customers) if service is None else service(n_customers, rng)
    D = np.full(n_customers, np.nan); start = np.full(n_customers, np.nan)
    free = [0.0] * c          # 각 서버가 비는 시각 (min-heap)
    in_system = []            # 시스템 내 손님의 퇴장 시각 (min-heap), K 검사용
    for i in range(n_customers):
        t = A[i]
        if K is not None:
            while in_system and in_system[0] <= t:
                heapq.heappop(in_system)
            if len(in_system) >= K:
                continue      # 차단
        f = heapq.heappop(free); s = max(t, f); d = s + S[i]
        heapq.heappush(free, d); start[i] = s; D[i] = d
        if K is not None:
            heapq.heappush(in_system, d)
    return A, start, D

In [ ]:
# helper (spkit에 없어서 여기서 정의) — 03d와 동일
def batch_means_se(x, n_batches=20):
    """자기상관 있는 수열의 평균에 대한 SE (배치 평균)."""
    x = np.asarray(x, float); m = len(x) // n_batches
    b = x[: m * n_batches].reshape(n_batches, m).mean(axis=1)
    return b.std(ddof=1) / np.sqrt(n_batches)

In [ ]:
WARM = N_CUST // 10                                  # 빈 시스템에서 출발 → 앞 10%는 워밍업으로 버림

A3, s3, D3 = mmck_customers(LAM, MU, C_SRV, N_CUST, rng)            # M/M/3
waited = ((s3 - A3) > 1e-12)[WARM:].astype(float)
p_wait_hat, se_pwait = waited.mean(), batch_means_se(waited, 20)

A33, s33, D33 = mmck_customers(LAM, MU, C_SRV, N_CUST, rng, K=C_SRV)   # M/M/3/3 (손실)
blocked = np.isnan(D33)[WARM:].astype(float)
block_hat, se_block = blocked.mean(), batch_means_se(blocked, 20)

print(f"M/M/3   fraction who wait : {p_wait_hat:.4f} ± {se_pwait:.4f}   (Erlang C = {m['p_wait']:.4f})")
print(f"M/M/3/3 fraction blocked  : {block_hat:.4f} ± {se_block:.4f}   (Erlang B = {pi_333[3]:.4f})")

### R3 재유도: 검사 역설의 평균

**명제.** 비율 $\lambda$ 푸아송 과정에서 고정 시각 $t$를 덮는 간격의 길이 $L_t=A(t)+R(t)$ (나이 + 잔여수명)의 기댓값은
$$E[L_t]=\frac{2-e^{-\lambda t}}{\lambda}\ \xrightarrow{t\to\infty}\ \frac2\lambda .$$
이를 유도하고, 극한 $2/\lambda$가 간격 평균 $1/\lambda$의 **두 배**인 이유를 길이 편향(length bias)으로 설명하라. 보너스: 덮는 간격 길이의 극한 분포가 $\mathrm{Gamma}(2,\lambda)$ (밀도 $\lambda^2xe^{-\lambda x}$)임을 보이고, 이것이 $\mathrm{Exp}(\lambda)$ 밀도에 길이 $x$를 곱해 정규화한 것과 같음을 확인하라.

**가정이 왜 필요한가.**
- 독립 증분: $A(t)$는 $t$ 이전, $R(t)$는 $t$ 이후의 과정으로 결정되므로 둘이 독립 — 합의 분포(보너스)에 쓰인다.
- 무기억성: $t$ 이후 첫 도착까지의 시간 $R(t)\sim\mathrm{Exp}(\lambda)$, 마지막 도착이 언제였든 상관없다.
- 시각 0에 도착(원점)이 있다는 관례: $A(t)\le t$ (절단 지수분포). 이것이 $e^{-\lambda t}$ 항의 출처다.

✍️ 내 유도 (R3): (책을 덮고 여기에 $E[A(t)]$, $E[R(t)]$, 길이 편향 설명을 적으세요)

In [ ]:
# 여기에 풀이: λ=1, t=10일 때 E[L_t]의 값을 적어 두고 아래 코드 검사와 비교

<details><summary>기대 답안 (R3)</summary>

**잔여.** 무기억성으로 $R(t)\sim\mathrm{Exp}(\lambda)$, $E[R(t)]=1/\lambda$ — $1/(2\lambda)$가 **아니다**(아래 traps).

**나이.** 시간을 거꾸로 보면 $t$에서 뒤로 가장 가까운 도착까지의 거리도 $\mathrm{Exp}(\lambda)$이지만, 원점(시각 0)에서 잘린다: $A(t)=\min(\mathrm{Exp}(\lambda),t)$. $P(A(t)>s)=e^{-\lambda s}$ ($s<t$), 0 ($s\ge t$)이므로
$$E[A(t)]=\int_0^tP(A(t)>s)\,ds=\int_0^te^{-\lambda s}ds=\frac{1-e^{-\lambda t}}{\lambda}.$$

**합.** $E[L_t]=\frac1\lambda+\frac{1-e^{-\lambda t}}{\lambda}=\frac{2-e^{-\lambda t}}{\lambda}$. $\lambda=1,t=10$: $2-e^{-10}\approx1.9999546$.

**길이 편향.** 간격들은 i.i.d. $\mathrm{Exp}(\lambda)$이지만 고정 시각 $t$는 **긴 간격에 걸릴 확률이 길이에 비례**해 크다. 길이 $x$인 간격이 뽑힐 확률 $\propto x$이므로 덮는 간격의 밀도는 $\propto x\cdot\lambda e^{-\lambda x}$, 정규화하면 $\lambda^2xe^{-\lambda x}$ — 이것이 $\mathrm{Gamma}(2,\lambda)$이고 평균 $2/\lambda$.

**보너스 (두 관점의 일치).** $t\to\infty$이면 $A,R$은 독립 $\mathrm{Exp}(\lambda)$이므로 합은 $\mathrm{Gamma}(2,\lambda)$ (지수 합성곱: $\int_0^x\lambda e^{-\lambda s}\lambda e^{-\lambda(x-s)}ds=\lambda^2xe^{-\lambda x}$). 길이 편향에서 얻은 밀도와 정확히 같다. 유한 $t$에서는 $A\le t$ 절단으로 $P(A=t)=e^{-\lambda t}$의 원자가 생기지만, $\lambda t=10$이면 $4.5\times10^{-5}$ 수준이라 히스토그램에서는 보이지 않는다.

</details>

코드 검사 — 시뮬레이션: 비율 1 푸아송 과정을 $(0,30]$ 위에 `N_REP`번 만들고 $t=10$에서 `age_residual_at`으로 $(A,R)$을 읽는다. 각 반복은 **독립**이므로 이번에는 i.i.d. SE(`mc_estimate`)가 정당하다. $(10,30]$에 도착이 하나도 없어 $R=\infty$인 경우(확률 $e^{-20}\approx2\times10^{-9}$)는 제외한다.

In [ ]:
LAM3, T3, HORIZON = 1.0, 10.0, 30.0
age = np.empty(N_REP); res = np.empty(N_REP)
for i in range(N_REP):
    times = poisson_arrivals(LAM3, HORIZON, rng)
    age[i], res[i] = age_residual_at(times, T3)
ok = np.isfinite(res)
age, res = age[ok], res[ok]
cover = age + res

cover_exact = (2 - np.exp(-LAM3 * T3)) / LAM3        # (2 - e^{-λt}) / λ
res_exact = 1 / LAM3
age_exact = (1 - np.exp(-LAM3 * T3)) / LAM3
r3_cover, r3_res, r3_age = mc_estimate(cover), mc_estimate(res), mc_estimate(age)
print(f"dropped (R = inf): {int((~ok).sum())} of {N_REP}")
print(f"E[A + R] : {r3_cover}   exact {cover_exact:.7f}")
print(f"E[R]     : {r3_res}   exact {res_exact:.7f}")
print(f"E[A]     : {r3_age}   exact {age_exact:.7f}")
print(f"sample variance of A + R: {cover.var(ddof=1):.3f}   (Gamma(2,1) variance = 2)")

히스토그램: 덮는 간격 길이 대 $\mathrm{Gamma}(2,1)$ 밀도 $xe^{-x}$. 길이 편향 관점과 "독립 지수 둘의 합" 관점이 같은 곡선을 준다.

In [ ]:
x = np.linspace(0, 12, 400)
pdf_gamma2 = LAM3 ** 2 * x * np.exp(-LAM3 * x)
ax = plot_hist_vs_pdf(cover, x, pdf_gamma2, bins=60, label_pdf=r"Gamma(2, 1) pdf  $x e^{-x}$")
ax.axvline(cover_exact, color="C3", ls="--", lw=1, label=f"exact mean 2 - e^-10 = {cover_exact:.5f}")
ax.set(xlabel="length of the interval covering t = 10", ylabel="density",
       title=f"Inspection paradox: covering interval (n = {cover.size})")
ax.legend();

## 4. Compare — 재유도 값 · 시뮬레이션 비교

정확값은 모두 위 코드가 공식에서 계산한 것이다(손으로 구한 분수를 그대로 적어 넣은 것이 아니다). 시뮬레이션 열은 3절의 결과다. "내 예측" 열에는 3절의 ✍️ 칸에 적은 **손 계산** 값을 넣어 보세요 (기본값 `None`).

In [ ]:
hand = {"R1_p01_at_half": None, "R2_p_wait_c3": None, "R2_erlang_b_3_2": None,
        "R3_cover_length": None, "R3_residual_mean": None}   # 손 계산 값을 적어 넣으세요

rows = [
    {"name": "R1_p01_at_half",   "predicted": hand["R1_p01_at_half"],   "estimate": r1_hat,     "exact": p01_exact},
    {"name": "R2_p_wait_c3",     "predicted": hand["R2_p_wait_c3"],     "estimate": p_wait_hat, "exact": m["p_wait"], "se": se_pwait},
    {"name": "R2_erlang_b_3_2",  "predicted": hand["R2_erlang_b_3_2"],  "estimate": block_hat,  "exact": pi_333[3],   "se": se_block},
    {"name": "R3_cover_length",  "predicted": hand["R3_cover_length"],  "estimate": r3_cover,   "exact": cover_exact},
    {"name": "R3_residual_mean", "predicted": hand["R3_residual_mean"], "estimate": r3_res,     "exact": res_exact},
]
Markdown(compare_table(rows))

각 행이 어긋나면 무엇을 뜻하는가:

- **R1_p01_at_half** — 어긋나면 질레스피 구현(체류 `1/q_i` 관례)이나 격자 인덱스 문제. `expm`과 닫힌 형식은 이미 $10^{-10}$에서 일치했으므로 유도가 아니라 시뮬레이션 쪽이다.
- **R2_p_wait_c3** — 어긋나면 워밍업 부족(빈 시스템 출발), 배치 SE가 아닌 i.i.d. SE, 또는 `mmc_metrics`의 $\rho=a/c$ 관례 오해.
- **R2_erlang_b_3_2** — 어긋나면 차단 판정(시스템 내 손님 수 $\ge K$)의 시점 처리 오류. 4/19가 서비스 분포에 무감각하다는 점을 기억.
- **R3_cover_length** — 어긋나면 `age_residual_at`의 관례(도착이 없으면 $A=t$)나 $R=\infty$ 처리. 절단 효과는 $10^{-5}$라 보이지 않는다.
- **R3_residual_mean** — 어긋나면 무기억성이 깨진 것 — 푸아송이 아닌 도착(예: 균등 간격)이라면 $1/\lambda$가 아니다.

In [ ]:
assert_close(r1_hat, p01_exact, k=4, name="R1_p01_at_half")
assert_close(p_wait_hat, m["p_wait"], se=se_pwait, k=4, name="R2_p_wait_c3")
assert_close(block_hat, pi_333[3], se=se_block, k=4, name="R2_erlang_b_3_2")
assert_close(r3_cover, cover_exact, k=4, name="R3_cover_length")
assert_close(r3_res, res_exact, k=4, name="R3_residual_mean")
print("all MC-vs-exact checks passed (k = 4)")

## 5. 자기채점 (Self-Grading)

복습의 목적은 '무엇이 보존되는가'를 한 줄로 말할 수 있는지 확인하는 것이다: 푸아송은 세분화·중첩·조건화에 닫혀 있고(02), CTMC는 흐름 균형 $\pi Q=0$으로 오랜 시간 뒤의 모습이 정해지며(03), 대기행렬은 그 특수해에 회계 항등식(리틀)과 관찰 원리(PASTA)를 얹은 것이다. 아래 표를 채운 뒤 X가 몰린 노트북부터 다시 연다.

| 문항 | 주제 (노트북) | 내 답 (O / △ / X) | 다시 볼 곳 |
|---|---|---|---|
| Q1 | 무기억성 (02a) | | 02a 정리 1 |
| Q2 | 지수 경주 (02a) | | 02a 정리 2 / 03a 정리 1 |
| Q3 | 푸아송 세 정의·순서통계량 (02b) | | 02b |
| Q4 | 세분화·중첩·NHPP (02c) | | 02c |
| Q5 | 복합 푸아송·검사 역설 (02d) | | 02d |
| Q6 | 생성원·체류·점프 (03a) | | 03a 정리 1 |
| Q7 | 콜모고로프·expm (03b) | | 03b 정리 2 |
| Q8 | 정상분포·수렴 (03b/03c) | | 03c 정리 1 / 03b 정리 3 |
| Q9 | 출생-사망 곱 형태 (03c) | | 03c 정리 2 |
| Q10 | M/M/1·리틀·PASTA (03d) | | 03d 정리 1-3 |
| Q11 | Erlang C/B·빠른1 vs 느린2 (03e) | | 03e 정리 1-2, 4 |
| Q12 | 균일화 (03f) | | 03f 정리 1, 3 |
| R1 | 2상태 P(t) 재유도 | | 03a 정리 2 |
| R2 | Erlang C/B 재유도 | | 03e 정리 1-2 |
| R3 | 검사 역설 평균 재유도 | | 02d |

**채점 규칙.** O = 책 없이 정확히, △ = 핵심은 맞았으나 상수·조건 누락, X = 못 함. 같은 노트북에서 X가 2개 이상이면 그 노트북의 이론·Predict 셀을 다시 풀고, △는 정리 카드만 다시 읽는다. `log/progress.md`에 `03r | O:__ △:__ X:__ | 다시 볼 노트북` 한 줄을 남긴다.

**자주 틀리는 곳 (△의 단골).** 자기채점에서 △를 준 문항이 아래 중 하나에 해당하는지 확인하세요.

- 체류시간을 `rng.exponential(q)`로 — 평균 $1/q$ 대신 비율 $q$를 넣음.
- 점프 사슬의 정상분포 $\nu$를 CTMC 정상분포로 착각 — $\pi_i\propto\nu_i/q_i$ (빨리 떠나는 상태는 덜 머문다).
- `np.exp(Q*t)` (원소별) vs `expm(Q*t)` (행렬 지수).
- CTMC 수렴에 주기성 조건을 요구하거나, 반대로 DTMC 수렴에서 비주기성을 빠뜨림.
- Erlang C의 $\rho$를 $\lambda/\mu$로 씀 (정답 $a/c$); $L=L_q+a$를 $L_q+c$로 씀; 유한 용량에서 리틀에 $\lambda$ 대신 $\lambda_{\rm eff}=\lambda(1-B)$를 써야 함을 잊음.
- 자기상관 표본(손님 체류시간, 시간평균)의 SE를 i.i.d.처럼 계산 — 배치 평균 또는 독립 반복.
- 검사 역설에서 잔여수명 평균을 $1/(2\lambda)$로 답함 (정답 $1/\lambda$; 무기억성).

## 6. 정리

- 푸아송 과정은 세분화·중첩·조건화(균등 순서통계량)에 닫혀 있고, 그 뿌리는 지수분포의 무기억성과 경쟁하는 시계다.
- CTMC = 체류 $\mathrm{Exp}(q_i)$ + 점프 $q_{ij}/q_i$; $P(t)=e^{Qt}$; 정상분포는 흐름 균형 $\pi Q=0$; 기약 유한이면 주기 조건 없이 수렴.
- 출생-사망 곱 형태 하나에서 M/M/1, M/M/∞, M/M/c, M/M/c/K, Erlang C/B가 모두 나온다.
- 리틀의 법칙은 회계 항등식, PASTA는 관찰 원리 — 둘을 합치면 시간평균 지표와 손님평균 지표가 연결된다.
- 균일화는 CTMC를 '푸아송 시계 + DTMC'로 분리해 계산·시뮬레이션·무주기성 증명을 한 번에 준다.

**Trap 카드.**
Q: 시뮬레이션 결과가 닫힌 형식과 3SE 밖에 있다. 가장 먼저 의심할 것 세 가지는?
A: (1) SE 자체 — 자기상관 표본을 i.i.d.로 취급했는가(배치 평균/독립 반복으로 다시 계산), (2) 매개변수 관례 — `rng.exponential`에 평균 대신 비율을 넣었는가, Erlang C의 $\rho=a/c$인가, (3) 가정 위반 — 정상 시작이 아닌데 정상값과 비교했는가(워밍업), 푸아송이 아닌 도착에 PASTA를 썼는가.

**다음 노트북: C2 (캡스톤 2 — 대기행렬 설계).** 03c–03e의 Erlang C와 손님 단위 시뮬레이션(`mmck_customers`)을 그대로 재사용해 SLA $P(W_q>t)\le\alpha$를 만족하는 최소 서버 수를 정하고, 비지수 서비스(M/G/c)에서 Erlang C의 강건성을 시험한다. 자기채점표에서 03d/03e에 X가 있으면 C2 전에 그 노트북을 먼저 다시 연다.

`log/progress.md` 한 줄 템플릿: `2026-__-__ | 03r | O:__ △:__ X:__ | 다시 볼 노트북: __ | 메모: __`